In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isocaproicacid,116.16,4.21197,3.507615095,259.4923136,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isocaproicacid,H,isocaproicacid,e,2000.0,0.03
"""

model = PCSAFT(["isocaproicacid"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isocaproicacid.csv")
fix_line_endings("rhol_isocaproicacid.csv")

Fixed: satp_isocaproicacid.csv
Fixed: rhol_isocaproicacid.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isocaproicacid.csv",
        "satp_isocaproicacid.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.7762669292008482


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2445.491985893754, 0.022218218318150806], PCSAFT{BasicIdeal, Float64}("isocaproicacid"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isocaproicacid.csv",   my_saturation_p)


=== AAD: satp_isocaproicacid.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
382.7500    3119.740000   3133.869289   0.4529  
388.8500    4226.320000   4246.053928   0.4669  
390.7500    4679.620000   4655.409519   0.5174  
393.8500    5412.890000   5396.062889   0.3109  
398.5500    6719.450000   6710.601734   0.1317  
402.6500    8039.340000   8071.476953   0.3997  
405.7500    9452.560000   9250.111971   2.1417  
407.5500    10079.170000  9999.159747   0.7938  
408.9500    10705.790000  10616.679917  0.8324  
412.0500    12119.000000  12099.574666  0.1603  
412.1500    12225.660000  12150.177079  0.6174  
414.3500    13238.910000  13309.427823  0.5327  
414.4500    13385.570000  13364.258329  0.1592  
420.3500    16878.610000  16953.608023  0.4443  
424.9500    20051.680000  20282.952652  1.1534  
426.8500    21704.880000  21808.801805  0.4788  
432.5500    26664.470000  26970.378039  1.1472  
439.0500    33157.270000  34051.454164  2.6968  
444.0500    40330.020000  40484.213272  0.3823  
447.7500   

0.6946694494691806

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isocaproicacid.csv",   my_liquid_density)


=== AAD: rhol_isocaproicacid.csv ===
Clapeyron Estimator  exp           calc          ARD%    
273.1500    937.100000    920.637651    1.7567  
293.1400    920.900000    904.755617    1.7531  
313.1300    904.100000    889.216338    1.6462  
333.1200    887.600000    873.797715    1.5550  
353.1200    870.400000    858.297666    1.3904  
373.1200    853.700000    842.549874    1.3061  
393.1300    836.700000    826.386143    1.2327  
413.1300    819.100000    809.670490    1.1512  
433.1400    801.100000    792.235610    1.1065  
453.1400    782.400000    773.943159    1.0809  
473.1500    763.400000    754.605313    1.1520  
493.1600    743.800000    734.036831    1.3126  
513.1700    723.000000    712.003757    1.5209  
533.1700    701.100000    688.222038    1.8368  
553.1800    674.400000    662.272992    1.7982  
AARD = 1.4400%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


1.4399668981371465